# Topic 05 · Audio tokenization: a miniature MOSS-Audio-Tokenizer, from waveform to tokens and back to speech

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd05-token-hoa-am-thanh-en.ipynb)

This notebook accompanies [Topic 05 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/), a deep-learning course in Vietnamese. It is the
English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd05-token-hoa-am-thanh.ipynb)):
the code, the data and every check are the same.

Language models can only read and write **tokens**, integers from a vocabulary (Lesson 14). To make one listen and
speak, you need an **audio tokenizer**: something that turns a waveform into a short sequence of integers, and
rebuilds sound from that sequence. The paper [*MOSS-Audio-Tokenizer*](https://arxiv.org/abs/2602.10934) (MOSI.AI and
Fudan University, 2026) proposes CAT (*Causal Audio Tokenizer with Transformer*): only causal Transformer blocks, no
convolutions at all, every part trained at once, then scaled up to 1.6 billion parameters and 3 million hours of
audio. On its tokens the authors build a fully autoregressive text-to-speech (TTS) model.

We don't have 3 million hours of audio, so we rebuild **the same design at a size that runs on a CPU**: 3,000
recordings of spoken English digits, a CAT of about a million parameters, and a small TTS that "reads" digits. For
each of the paper's claims we run the experiment and conclude *holds*, *does not hold* or *depends*, as the course's
Topics 02 and 03 do.

**What you'll do:**
1. Count bits: from the waveform's 128,000 bits per second down to 125 bits per second for one layer of tokens.
2. See why comparing sample by sample (MSE) is a poor measure for audio, then write the paper's multi-scale mel loss.
3. Write residual vector quantization (RVQ) and *quantizer dropout* yourself, and see why a tokenizer has to survive
   having layers cut off.
4. Build a miniature CAT, check that it can stream, then listen to a digit through 1, 2, 4 … 32 layers of tokens.
5. Test the paper's claims about training and scaling, plus two things the paper doesn't try.
6. Build a miniature CAT-TTS (Temporal + Depth Transformer) and test *Progressive Sequence Dropout*.

**Background** (the course's lessons, in Vietnamese):
[Lesson 14 · Tokenization & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/),
[Lesson 17 · Transformer (GPT-2)](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) (causal Transformer
blocks), [Lesson 18 · Vision Transformer](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vision-transformer/)
(cutting into patches), [Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/)
and [Lesson 20 · GANs](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gan/).
**Run time:** about a minute on a CPU, including the first data download.

The notebook uses **PyTorch**. Each section runs a small version live, then loads the full results (3 seeds) that
the course's script `reel/codec/sweep.py` computed in advance, with the same code and the same seeds. Some cells play
audio: put on headphones and *listen*, because the human ear is still the best judge of an audio codec.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

In [ ]:
import base64
import hashlib
import io
import json
import math
import os
import tarfile
import urllib.request
import wave
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number with thousands separators: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


def nlayers(k):
    return f"{k} layer" if k == 1 else f"{k} layers"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 10000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def play(x, label=""):
    """Play an 8 kHz clip in the notebook (loudness kept, clipped to [−1, 1])."""
    if label:
        print(label)
    display(Audio(np.clip(np.asarray(x, np.float64), -1, 1), rate=SR, normalize=False))

## 2 · Data: 3,000 spoken digits

The [Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD, version 1.0.10,
CC BY-SA 4.0): 6 speakers, each reading each digit from 0 to 9 fifty times, recorded at 8,000 samples per second
(8 kHz), 16 bits per sample. The archive comes from GitHub; we check the MD5 of the *contents* of the 3,000 wav files
rather than of the archive, because GitHub can recompress an archive without changing what is in it. Every recording
is scaled to the same loudness (root mean square, RMS, of 0.1), so loudness can't give away who is speaking. We split
by recording index as the dataset's README says: recordings 0–4 for testing, 5–9 for validation (choosing
hyperparameters), the rest for training. Four recordings longer than 1.28 seconds are dropped (three from training,
one from validation), so the test set keeps all 300.

In [ ]:
SR, HOP = 8000, 640  # 8 kHz; one token frame = 640 samples = 80 ms, i.e. 12.5 frames per second as in the paper
WIN = 16  # at most 16 frames per clip: 16 · 80 ms = 1.28 s
SPEAKERS = ("george", "jackson", "lucas", "nicolas", "theo", "yweweler")
FSDD = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz"
TGZ = Path("fsdd-v1.0.10.tar.gz")
if not TGZ.exists():
    urllib.request.urlretrieve(FSDD, TGZ)


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())

In [ ]:
raw = {}
with tarfile.open(TGZ) as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".wav") and "/recordings/" in m.name:
            raw[m.name.rsplit("/", 1)[1]] = tf.extractfile(m).read()
h = hashlib.md5()
for k in sorted(raw):
    h.update(raw[k])
assert len(raw) == 3000 and h.hexdigest() == "2c93442f2c808e24d4d913f6be24d32c", "FSDD is corrupted; delete it"

waves, digit, spk, idx = [], [], [], []
for name in sorted(raw):
    d, s, i = name[:-4].split("_")
    with wave.open(io.BytesIO(raw[name])) as w:
        assert (w.getnchannels(), w.getsampwidth(), w.getframerate()) == (1, 2, SR)
        x = np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float64) / 32768
    waves.append(np.clip(x * 0.1 / np.sqrt(np.mean(x**2)), -1, 1).astype(np.float32))
    digit.append(int(d))
    spk.append(SPEAKERS.index(s))
    idx.append(int(i))
digit, spk, idx = np.array(digit), np.array(spk), np.array(idx)
n_samples = np.array([len(x) for x in waves])
keep = n_samples <= WIN * HOP
split = np.where(idx < 5, "test", np.where(idx < 10, "val", "train"))
TRAIN, VAL, TEST = (np.where((split == s) & keep)[0] for s in ("train", "val", "test"))
dur = n_samples / SR
print(f"{num(len(waves), 0)} recordings, {num(dur.sum() / 60, 1)} minutes; {num(dur.min(), 2)}–{num(dur.max(), 2)} s "
      f"long, median {num(np.median(dur), 2)} s")  # fmt: skip
print(f"training {num(len(TRAIN), 0)} · validation {len(VAL)} · test {len(TEST)}")
assert (len(TRAIN), len(VAL), len(TEST)) == (2397, 299, 300)

A recording is a sequence of real numbers, each one the air pressure at one instant. The figure below shows the word
"seven" from one speaker in the test set. On the left is the waveform (time runs left to right). On the right is the
**spectrogram**: time across, frequency up, and brighter means more energy. The parallel, evenly spaced horizontal
lines are **harmonics**: multiples of the pitch of the voice. Their spacing tells you how low or high the voice is,
a large part of "whose voice". The broader bright bands (formants) tell you which vowel the mouth is making, that
is, "what is said".

In [ ]:
EX = TEST[(digit[TEST] == 7) & (spk[TEST] == SPEAKERS.index("theo"))][0]
x_ex = waves[EX]


def spectrogram(ax, x, title, win=256, vmin=-4):
    S = torch.stft(torch.tensor(x), win, win // 4, window=torch.hann_window(win), return_complex=True)
    S = S.abs().clamp_min(1e-5).log10().numpy()
    ax.imshow(S, origin="lower", aspect="auto", vmin=vmin, vmax=1, cmap="magma", extent=[0, len(x) / SR, 0, SR / 2])
    ax.set(title=title, xlabel="time (s)", ylabel="frequency (Hz)")


fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
t = np.arange(len(x_ex)) / SR
a1.plot(t, x_ex, color=BLUE, lw=0.6)
a1.set(title='waveform: "seven"', xlabel="time (s)", ylabel="amplitude")
spectrogram(a2, x_ex, "spectrogram (log₁₀ magnitude)")
fmt_axes(fig)
plt.tight_layout()
plt.show()
play(x_ex, f'digit {digit[EX]}, speaker "{SPEAKERS[spk[EX]]}", recording {idx[EX]}')

## 3 · What the paper says, and what we check

CAT has three parts, one after another (the paper's Figure 2):

- The **encoder** cuts the waveform into patches of 240 samples, passes them through *causal* Transformer blocks
  (each position only looks at the past), and three times merges each pair of neighbouring positions into one
  (*patchify*, like cutting an image into patches in Lesson 18). 24,000 samples per second become 12.5 vectors per
  second.
- The **quantizer** is an RVQ with 32 layers. Each layer picks one of 1,024 codes, so each vector becomes 32
  integers: those are the tokens.
- The **decoder** mirrors the encoder, is also causal, and rebuilds the waveform from the tokens.

All three are trained together on one total loss: a multi-scale mel loss (section 5), the quantizer's two losses
(section 6), a *semantic* loss (a language model with 0.5 billion parameters reads the quantizer's output and has to
write out the speech or a description of the sound) and a GAN loss (Lesson 20). Our miniature keeps that shape:

| | Paper | Here |
|---|---|---|
| Data | about 3 million hours of speech, sound and music; 24 kHz | 21.9 minutes of English digits; 8 kHz |
| Patching | 240 · 2 · 2 · 2 = 1,920 samples → 12.5 frames/s | 80 · 2 · 2 · 2 = 640 samples → 12.5 frames/s |
| Encoder + decoder | 2 × 68 Transformer blocks, width 768–1,280; 1.6 billion parameters | 2 × 8 blocks, width 64 |
| RVQ | 32 layers × 1,024 codes, lookup in 8 dimensions | the same |
| Mel loss | windows of 2⁵…2¹¹ samples | the same (at 8 kHz each window lasts 3 times as long) |
| Semantic | 0.5B LLM; speech recognition, audio captioning | a 2-block model that reads out the digit |
| Decoder output | waveform samples | the spectrum of each frame, then an inverse Fourier transform (section 7) |

From the paper's experiments we take six claims that can be tested at small scale:

| | Claim | Evidence in the paper | Section |
|---|---|---|---|
| A | *Quantizer dropout* gives a tokenizer that works at every bitrate | rate 1 only, no comparison | 6, 8 |
| B | End-to-end training keeps improving; "partial" training (encoder and quantizer frozen) stalls early | Fig. 4 | 8 |
| C | Bigger models are better, especially at high bitrates; at low bitrates the bits are the bottleneck | Fig. 5 | 8 |
| D | Larger batches are better at every point of training, for the same number of steps | Figure 6 | 8 |
| E | The semantic loss makes the tokens carry meaning | no ablation | 8 |
| F | *Progressive Sequence Dropout* keeps TTS good when fewer layers are generated; without it, quality drops | Fig. 3 | 9 |

The paper doesn't try other quantizer dropout rates (A) and doesn't remove the semantic loss (E); we try both.

## 4 · Counting bits

A wav file at 8 kHz and 16 bits costs 8,000 · 16 = 128,000 bits per second. A token is a number in 0…1,023, that is
log₂ 1,024 = 10 bits. At 12.5 frames per second, each RVQ layer costs 12.5 · 10 = 125 bits per second.

> **🤔 Predict first:** with all 32 layers, how many bits is one second of speech, and how many times smaller is that
> than the wav file? If all 32 layers were laid out in one flat sequence for a language model to read, how many
> tokens long would one second be?

In [ ]:
FRAME_RATE = SR / HOP
NQ, CODES = 32, 1024
BITS = math.log2(CODES)
print(f"{num(FRAME_RATE, 1)} frames/s · {num(BITS, 0)} bits per token · {num(FRAME_RATE * BITS, 0)} bits/s per layer")
print(f"{'layers':>6s} | {'bits/s':>7s} | {'× smaller than 8 kHz wav':>24s} | {'× smaller than 24 kHz wav (paper)':>33s} | "
      "tokens/s if flattened")  # fmt: skip
for k in (1, 2, 4, 8, 16, 32):
    bps = FRAME_RATE * k * BITS
    print(f"{k:6d} | {num(bps, 0):>7s} | {num(SR * 16 / bps, 0):>24s} | {num(24000 * 16 / bps, 0):>33s} | "
          f"{num(FRAME_RATE * k, 1):>21s}")  # fmt: skip
assert FRAME_RATE * NQ * BITS == 4000

All 32 layers make 4,000 bits per second (4 kbps): 32 times smaller than a wav file at 8 kHz, and 96 times smaller
than one at the paper's 24 kHz. Flattened, one second becomes 400 tokens, so a 10-second utterance is already 4,000
tokens. That is why CAT-TTS (section 9) doesn't flatten: one Transformer runs over time and sees only 12.5 positions
per second, and a second, small Transformer handles the 32 layers inside each frame.

How many of those bits are *content*? Each of our recordings is one digit, that is log₂ 10 ≈ 3.3 bits, in about
0.42 seconds (the median): under 8 bits per second. One layer of tokens (125 bits/s) is already about 15 times that.
The rest goes to *whose voice*, intonation, breath and noise: things a text-to-speech model has to generate too.

## 5 · Measuring the distance between two sounds

The decoder rebuilds the waveform, and then we need a loss that says how "close" the rebuild is to the original. The
most direct choice is the mean squared error (MSE) between each pair of samples, as with pixels in Lesson 19. Below
are five variants of the same "seven": the whole wave with its sign flipped (+x becomes −x), a 1 ms delay, every
frequency's magnitude kept but its phase scrambled at random, 20 dB of white noise added, and every frequency above
2 kHz removed.

> **🤔 Predict first:** listen to the five variants in the cell below and rank them by how different they sound from
> the original. In what order will MSE rank them?

In [ ]:
x0 = torch.tensor(x_ex)[None]
g = torch.Generator().manual_seed(0)
S0 = torch.stft(x0, 256, 64, window=torch.hann_window(256), return_complex=True)
phase = torch.exp(2j * torch.pi * torch.rand(S0.shape, generator=g))
X0, f_hz = torch.fft.rfft(x0), torch.fft.rfftfreq(x0.shape[1], 1 / SR)
variants = {
    "sign flip": -x0,
    "1 ms delay": torch.roll(x0, SR // 1000, 1),
    "phase scramble": torch.istft(S0.abs() * phase, 256, 64, window=torch.hann_window(256), length=x0.shape[1]),
    "20 dB noise": x0 + torch.randn(x0.shape, generator=g) * x0.std() * 0.1,
    "low-pass 2 kHz": torch.fft.irfft(X0 * (f_hz < 2000), n=x0.shape[1]),
}
for name, y in variants.items():
    play(y[0], name)

The sign flip and the 1 ms delay sound *identical* to the original: the ear can't hear the sign of the pressure, and
1 ms is shorter than any time interval the ear can tell apart. The scrambled phase sounds a little rough. The noise
is a clear hiss, and removing everything above 2 kHz makes the voice sound muffled, like an old telephone. Now MSE
(divided by the power of the original, so 1 means an error as large as the whole signal):

In [ ]:
power = (x0**2).mean()
mse = {name: float(((y - x0) ** 2).mean() / power) for name, y in variants.items()}
for name, v in mse.items():
    print(f"{name:16s} MSE / power = {num(v, 3)}")
assert abs(mse["sign flip"] - 4) < 1e-6 and min(mse, key=mse.get) == "20 dB noise"

MSE ranks them the opposite way from the ear. The two variants that sound identical to the original are punished
hardest (the sign flip is off by 4 times the whole signal, because (−x − x)² = 4x²), and the clearly audible noise is
barely punished at all. MSE compares sample by sample, so it also demands the right **phase**, the exact position of
every oscillation, while the ear mostly hears **the magnitude of each frequency over time**. A decoder trained with
MSE has to guess the phase. When it isn't sure, it returns the average of the possible phases, and the average of
oscillations that are out of phase is close to 0: the sound gets quiet and dull.

So the paper, like DAC [27] before it, measures distance on the **mel spectrum**:

1. A short-time Fourier transform (STFT): cut the wave into overlapping windows (*w* samples long, *w*/4 apart),
   multiply each by a Hann window, and take the *magnitude* of each frequency. The phase is thrown away.
2. Group the frequencies into bands on the **mel** scale: narrow at low frequencies and wider at high ones, like the
   resolution of the ear.
3. Take log₁₀ (after clamping at 10⁻⁵): the ear hears loudness in ratios, so doubling the energy is the same step
   everywhere.
4. Take the mean absolute difference (L1), and add it up over 7 window lengths *w* = 32, 64, …, 2,048 samples, with
   5, 10, …, 320 mel bands.

We write it ourselves:

In [ ]:
MEL_WINDOWS = tuple(2**i for i in range(5, 12))  # 32 … 2,048 samples
MEL_BANDS = (5, 10, 20, 40, 80, 160, 320)


def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, np.float64) / 700)


def mel_filters(n_fft, n_mels):
    """Triangular filters on the mel scale, (n_fft/2 + 1, n_mels); every filter has the same area."""
    freqs = np.linspace(0, SR / 2, n_fft // 2 + 1)
    edges = 700 * (10 ** (np.linspace(hz_to_mel(0), hz_to_mel(SR / 2), n_mels + 2) / 2595) - 1)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    tri = np.maximum(0, np.minimum((freqs[None] - lo) / (mid - lo), (hi - freqs[None]) / (hi - mid)))
    return torch.tensor((tri * (2 / (hi - lo))).T, dtype=torch.float32)


_FB = {}


def mel(x, win, n_mels):
    """|STFT| (Hann window of length win, hop win/4) → the magnitude in each mel band, (B, frames, n_mels)."""
    if (win, n_mels) not in _FB:
        _FB[win, n_mels] = (torch.hann_window(win), mel_filters(win, n_mels))
    window, fb = _FB[win, n_mels]
    s = torch.stft(x, win, win // 4, window=window, center=True, return_complex=True).abs()
    return s.transpose(1, 2) @ fb


def mel_loss(x, y):
    """Σ over the 7 window lengths of the mean |log₁₀ mel(x) − log₁₀ mel(y)| (the paper's Eq. 4, computed as DAC does)."""
    total = 0
    for win, n_mels in zip(MEL_WINDOWS, MEL_BANDS, strict=True):
        a = mel(x, win, n_mels).clamp_min(1e-5).log10()
        b = mel(y, win, n_mels).clamp_min(1e-5).log10()
        total = total + (a - b).abs().mean()
    return total


mels = {name: float(mel_loss(y, x0)) for name, y in variants.items()}
print(f"{'variant':16s} | {'MSE / power':>11s} | {'mel loss':>8s}")
for name in variants:
    print(f"{name:16s} | {num(mse[name], 3):>11s} | {num(mels[name], 2):>8s}")
assert mels["sign flip"] < 1e-6 and mels["1 ms delay"] < min(mels["20 dB noise"], mels["low-pass 2 kHz"])

The mel loss ranks them much as the ear does: the sign flip is exactly 0, the 1 ms delay close to 0, and removing
everything above 2 kHz is punished hardest. The scrambled phase is still punished fairly hard: random phases make the
overlapping windows partly cancel each other, so the magnitudes measured again are no longer what they were. So the
mel loss indirectly asks for phases that are *consistent* between neighbouring windows, though not for any
particular phase.

Why several window lengths? The figure below shows the same "seven" with the shortest and the longest window. The
32-sample window (4 ms) shows clearly *when* the sound starts and stops, but every frequency smears into vertical
streaks. The 2,048-sample window (256 ms) separates each harmonic into its own horizontal line, but smears in time. No
single window sees both, so the loss adds up all seven.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, win in zip(axes, (32, 2048), strict=True):
    spectrogram(ax, x_ex, f"window of {num(win, 0)} samples ({num(1000 * win / SR, 0)} ms)", win=win)
fmt_axes(fig)
plt.tight_layout()
plt.show()

## 6 · Residual vector quantization (RVQ)

The encoder outputs one real-valued vector per frame. To get tokens, each vector has to be replaced by an integer:
the index of the nearest code in a *codebook*, as in k-means, where each point belongs to its nearest centre. A
codebook of 1,024 codes gives only 10 bits per frame, far too coarse to rebuild speech. For 40 bits, a single
codebook would need 2⁴⁰ codes, over a trillion.

**RVQ** works differently: layer 1 quantizes the vector, layer 2 quantizes the *residual* (the vector minus the code
layer 1 chose), layer 3 quantizes what is left after that, and so on. Each layer needs only a small codebook, the
bits add up, and each layer corrects the errors of the layers before it. Here it is on two-dimensional points, with a
codebook of 8 codes per layer (3 bits), found by k-means:

In [ ]:
rng = np.random.default_rng(0)
pts = np.concatenate([rng.normal(c, 0.35, (400, 2)) for c in rng.uniform(-3, 3, (6, 2))])


def kmeans(P, k, seed, iters=30):
    r = np.random.default_rng(seed)
    c = P[r.choice(len(P), k, replace=False)]
    for _ in range(iters):
        a = ((P[:, None] - c[None]) ** 2).sum(-1).argmin(1)
        c = np.array([P[a == j].mean(0) if (a == j).any() else c[j] for j in range(k)])
    return c, a


res, rec, books, errs = pts.copy(), np.zeros_like(pts), [], []
for layer in range(6):
    c, a = kmeans(res, 8, seed=layer)
    books.append((res.copy(), c))
    rec += c[a]
    res = pts - rec
    errs.append(float((res**2).sum(1).mean()))
for layer, e in enumerate(errs, 1):
    print(f"{nlayers(layer):>8s} = {3 * layer:2d} bits: mean squared error {num(e, 4)}")
assert all(np.diff(errs) < 0)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (r, c), name in zip(axes[:2], books[:2], ("layer 1: the points themselves", "layer 2: residual after layer 1"),
                            strict=True):  # fmt: skip
    ax.scatter(r[:, 0], r[:, 1], s=3, color=BLUE, alpha=0.4, label="points to quantize")
    ax.scatter(c[:, 0], c[:, 1], s=60, marker="s", color=ORANGE, edgecolor="k", label="the layer's 8 codes")
    ax.set(title=name, aspect="equal")
axes[0].legend(loc="lower left", fontsize=8)
axes[2].plot(range(3, 19, 3), errs, "o-", color=GREEN)
axes[2].set(yscale="log", xlabel="total bits", ylabel="mean squared error", title="each layer adds 3 bits")
fmt_axes(fig)
plt.tight_layout()
plt.show()

Note the middle panel's scale: the points span about ±4, while the residual after layer 1 spans only about ±1. Layer 2
places its 8 codes around the origin, where the residual is concentrated. Each layer cuts the error several times. The
order matters: **the first layers carry the coarse part, the later layers refine it**. Cutting off the last layers
leaves a coarser approximation instead of breaking it. That is what the paper uses to choose the bitrate at run time:
keeping the first K layers costs 125·K bits per second.

### One RVQ layer of CAT

CAT doesn't run k-means. Each layer is a module trained by gradient descent, following DAC [27]:

- **Factorized**: project the 64-dimensional residual down to 8 dimensions, find the nearest code in those 8
  dimensions, then project that code back up to 64. Searching in a small space gets more of the codes used.
- **L2-normalized**: "nearest" is measured by cosine, that is after scaling both the vector and the codes to length 1.
  What gets projected back is the unnormalized code.
- Choosing a code (an argmax) has no derivative. **Straight-through**: in the backward pass, treat quantization as
  the identity, so the gradient at the output passes unchanged to the input. In code that is `ze + (zq − ze).detach()`:
  its value is `zq`, and its gradient goes to `ze`.
- Because straight-through skips the codebook, the codebook needs losses of its own. The paper's two (Eq. 2–3), with
  sg for stop-gradient: ℒ_cmt = ‖z − sg(q(z))‖² pulls the encoder's output towards the chosen code, and
  ℒ_code = ‖sg(z) − q(z)‖² pulls the code towards the encoder's output. Their weights are 0.25 and 1, as in VQ-VAE
  and in the paper.

In [ ]:
class VQLayer(nn.Module):
    """A DAC-style RVQ layer: project d → cdim, find the nearest code by cosine, project the (unnormalized) code back."""

    def __init__(self, d, codes=CODES, cdim=8):
        super().__init__()
        self.inp, self.outp = nn.Linear(d, cdim), nn.Linear(cdim, d)
        self.book = nn.Parameter(torch.randn(codes, cdim))

    def forward(self, r):
        ze = self.inp(r)
        idx = (F.normalize(ze.detach(), dim=-1) @ F.normalize(self.book, dim=-1).T).argmax(-1)
        zq = self.book[idx]
        cmt = ((ze - zq.detach()) ** 2).mean(-1)  # Eq. 2, per vector
        cb = ((zq - ze.detach()) ** 2).mean(-1)  # Eq. 3
        return self.outp(ze + (zq - ze).detach()), idx, cmt, cb


torch.manual_seed(0)
layer = VQLayer(64)
r = torch.randn(64)
q, code, cmt, cb = layer(r[None])
print(
    f"64-d vector → token {int(code[0])} (10 bits) → 64-d vector; ℒ_cmt = {num(cmt.item(), 3)}, "
    f"ℒ_code = {num(cb.item(), 3)}"
)
J = torch.autograd.functional.jacobian(lambda v: layer(v[None])[0][0], r)
assert torch.allclose(J, layer.outp.weight @ layer.inp.weight, atol=1e-6)
print("Jacobian of the output w.r.t. the input = W_out · W_in: the gradient passes through argmax as if nothing were "
      "quantized")  # fmt: skip

### Quantizer dropout: learning to be cut short

In the k-means example, each layer is found after the earlier layers are fixed, so the first layer naturally carries
the coarse part. When the encoder, the layers and the decoder are learned *together* with all 8 layers, nothing forces
the first layers to stand on their own: the network only needs the sum of all 8 to be right. **Quantizer dropout**
(SoundStream, 2021) fixes that: for each training example, pick a random number of layers K from 1 to 8 and use only
the first K. The paper applies it to *every* example (rate 1), as SoundStream does. DAC applies it to only half of
the examples, because applying it to every example lowers the quality when all the layers are kept.

Here it is on a small problem: compress each 40-band mel frame of FSDD (10 ms, voiced frames only) through an MLP
encoder, 8 `VQLayer`s with 16 codes per layer (4 bits), then an MLP decoder. Two identical models: one learns with
all 8 layers, the other with a random number of layers.

> **🤔 Predict first:** when only the first 1 or 2 layers are kept, which model makes the smaller error? And with
> all 8?

In [ ]:
def logmel(x):
    """(B, L) → (B, 40, frames): log₁₀ magnitude in 40 mel bands; 32 ms window, 10 ms hop."""
    if "logmel" not in _FB:
        _FB["logmel"] = (torch.hann_window(256), mel_filters(256, 40))
    window, fb = _FB["logmel"]
    s = torch.stft(x, 256, 80, window=window, center=True, return_complex=True).abs()
    return (s.transpose(1, 2) @ fb).clamp_min(1e-5).log10().transpose(1, 2)


def pad_clips(ids):
    out = np.zeros((len(ids), WIN * HOP), np.float32)
    for b, c in enumerate(ids):
        out[b, : len(waves[c])] = waves[c]
    return torch.tensor(out)


def mel_frames(ids):
    m = logmel(pad_clips(ids)).transpose(1, 2).reshape(-1, 40)
    return m[m.max(1).values > -3]  # drop the silent frames


F_TR, F_VA = mel_frames(TRAIN[:600]), mel_frames(VAL)
mu, sd = F_TR.mean(0), F_TR.std(0)
F_TR, F_VA = (F_TR - mu) / sd, (F_VA - mu) / sd


class ToyRVQ(nn.Module):
    def __init__(self, nq=8, codes=16, d=32, cdim=4):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(40, 128), nn.GELU(), nn.Linear(128, d))
        self.dec = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, 40))
        self.layers = nn.ModuleList(VQLayer(d, codes, cdim) for _ in range(nq))

    def forward(self, x, n_q):
        r = self.enc(x)
        out, vq = torch.zeros_like(r), 0
        for i, lay in enumerate(self.layers):
            m = (i < n_q).float()  # examples that use fewer layers get 0 from this one
            q, _, cmt, cb = lay(r)
            q = q * m[:, None]
            out, r = out + q, r - q
            vq = vq + ((0.25 * cmt + cb) * m).sum() / m.sum()  # the loss counts only the examples using this layer
        return self.dec(out), vq


def train_toy(qdrop, seed=0, steps=300, bs=256, nq=8):
    torch.manual_seed(seed)
    m = ToyRVQ(nq)
    opt = torch.optim.Adam(m.parameters(), 3e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        b = F_TR[torch.randint(len(F_TR), (bs,), generator=g)]
        drop = torch.rand(bs, generator=g) < qdrop
        n_q = torch.where(drop, torch.randint(1, nq + 1, (bs,), generator=g), torch.tensor(nq))
        y, vq = m(b, n_q)
        loss = F.mse_loss(y, b) + vq
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return [F.mse_loss(m(F_VA, torch.full((len(F_VA),), k))[0], F_VA).item() for k in range(1, nq + 1)]


toy = {qd: train_toy(qd) for qd in (0.0, 1.0)}
LABEL = {0.0: "trained with all 8 layers", 1.0: "trained with random layer counts"}
print("layers kept".ljust(34) + "".join(f"{k:>6d}" for k in range(1, 9)))
for qd, errs_k in toy.items():
    print(LABEL[qd].ljust(34) + "".join(f"{num(e, 2):>6s}" for e in errs_k))
assert toy[1.0][0] < toy[0.0][0] and toy[1.0][1] < toy[0.0][1] and toy[0.0][-1] < toy[1.0][-1]

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(range(1, 9), toy[0.0], "o-", color=ORANGE, label=LABEL[0.0])
ax.plot(range(1, 9), toy[1.0], "s--", color=BLUE, label=LABEL[1.0])
ax.set(xlabel="layers kept at use", ylabel="validation MSE", title="quantizer dropout, small problem")
ax.legend()
fmt_axes(fig)
plt.tight_layout()
plt.show()

The model that only ever learned with all 8 layers makes large errors when it is cut short (orange line): its first
layers were never taught to stand alone, and adding a layer sometimes even makes things *worse*. The model that
learned with random layer counts (blue line) does much better at 1–2 layers. In exchange it does *worse* at all 8
layers, and its line is nearly flat from layer 4 on: layer k is only trained on the examples that keep at least k
layers, that is (9 − k)/8 of them, so the last layers learn least. That is why DAC applies dropout to only half of
the examples. Section 8 measures this trade-off on the audio tokenizer itself, at rates 0, ½ and 1.

## 7 · A miniature CAT

Now we put everything together into a tokenizer. The encoder takes a clip of 1.28 seconds (10,240 samples):

1. Cut it into 128 patches of 80 samples (10 ms each), as ViT cuts an image into patches.
2. **Stage 1**: project each patch to 64 dimensions, pass it through 2 causal Transformer blocks, and project it
   down to 32 dimensions.
3. **Patchify**: merge each pair of neighbouring positions into one 64-dimensional vector, so 128 positions become 64
   (20 ms each).
4. Repeat twice more (stages 2 and 3): 64 → 32 → 16 positions. Stage 4 outputs 16 vectors of 64 dimensions, each for
   80 ms: exactly 12.5 frames per second.
5. The 32-layer RVQ turns each vector into 32 tokens.

The decoder goes the other way: it adds up the chosen codes and goes through 4 stages, each time splitting one
position into two (*unpatchify*). Every block uses causal attention (position t only looks at positions ≤ t),
encodes positions with RoPE as today's LLMs do, and multiplies each branch by a learned *layer scale* initialized at
0.01, as in the paper, so the deep network starts out close to the identity.

**One departure from the paper.** The paper's CAT directly outputs the 1,920 waveform samples of each frame. Ours
outputs, every 10 ms, the *spectrum* of a 40 ms window (the log magnitude and the phase of 161 frequencies), then
takes the inverse Fourier transform and overlap-adds the windows, as Vocos does (Siuzdak, 2023). In short pilot runs
(3,000 steps), outputting samples directly learned much more slowly at this size; section 8 compares the two on the
same 10,000-step budget. The output is still produced by Transformers with no convolution, but we pay a price: the
40 ms window spills 30 ms into the next frame, so the last 30 ms of each frame are only finished when the next
frame's tokens arrive. The paper has no such delay.

In [ ]:
PATCH = (80, 2, 2, 2)


def rope(t, dh):
    inv = 1.0 / (10000 ** (torch.arange(0, dh, 2, dtype=torch.float32) / dh))
    ang = torch.arange(t, dtype=torch.float32)[:, None] * inv[None]
    return ang.cos(), ang.sin()


def apply_rope(x, cs):
    cos, sin = cs
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack((a * cos - b * sin, a * sin + b * cos), -1).flatten(-2)


class Block(nn.Module):
    """Pre-LN Transformer block: causal attention (RoPE), then a 4× GELU MLP; each branch is scaled by a layer scale
    initialized at 0.01."""

    def __init__(self, d, heads=4, ls=0.01):
        super().__init__()
        self.h = heads
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.f1, self.f2 = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.s1 = nn.Parameter(torch.full((d,), ls))
        self.s2 = nn.Parameter(torch.full((d,), ls))

    def forward(self, x, cs):
        B, T, d = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, d // self.h).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cs), apply_rope(k, cs)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.s1 * self.o(a.transpose(1, 2).reshape(B, T, d))
        return x + self.s2 * self.f2(F.gelu(self.f1(self.n2(x))))


class Stage(nn.Module):
    """Input projection → causal blocks → output projection (MOSS's ProjectedTransformer)."""

    def __init__(self, d_in, d, layers, d_out, heads=4):
        super().__init__()
        self.inp, self.out = nn.Linear(d_in, d), nn.Linear(d, d_out)
        self.blocks = nn.ModuleList(Block(d, heads) for _ in range(layers))
        self.dh = d // heads

    def forward(self, x):
        cs = rope(x.shape[1], self.dh)
        x = self.inp(x)
        for b in self.blocks:
            x = b(x, cs)
        return self.out(x)


class RVQ(nn.Module):
    """32 VQLayers applied one after another to the residual."""

    def __init__(self, d, nq=NQ):
        super().__init__()
        self.layers = nn.ModuleList(VQLayer(d) for _ in range(nq))

    def encode(self, z, k):
        res, codes = z, []
        for lay in self.layers[:k]:
            q, idx, _, _ = lay(res)
            res = res - q
            codes.append(idx)
        return torch.stack(codes, -1)

    def decode(self, codes):
        return sum(lay.outp(lay.book[codes[..., i]]) for i, lay in enumerate(self.layers[: codes.shape[-1]]))


def overlap_add(frames, hop):
    """(B, T, n) → (B, T·hop): window t ends at sample (t + 1)·hop; whatever falls before time 0 is dropped."""
    B, T, n = frames.shape
    y = F.fold(frames.transpose(1, 2), (1, (T - 1) * hop + n), (1, n), stride=(1, hop))
    return y.reshape(B, -1)[:, n - hop : n - hop + T * hop]


TAG, BOS_T, EOS_T = 10, 11, 12  # text vocabulary: digits 0–9, task tag, begin, end


class SemanticLM(nn.Module):
    """The semantic objective in miniature: [task tag] + the RVQ output + the transcript; causal, one digit at a time."""

    def __init__(self, d_audio, d=64, layers=2):
        super().__init__()
        self.audio = nn.Linear(d_audio, d)
        self.emb = nn.Embedding(13, d)
        self.blocks = nn.ModuleList(Block(d) for _ in range(layers))
        self.norm, self.head = nn.LayerNorm(d), nn.Linear(d, 13)
        self.dh = d // 4

    def logits(self, q, text_in):
        B, T = q.shape[:2]
        h = torch.cat([self.emb(torch.full((B, 1), TAG)), self.audio(q), self.emb(text_in)], 1)
        cs = rope(h.shape[1], self.dh)
        for b in self.blocks:
            h = b(h, cs)
        return self.head(self.norm(h[:, 1 + T :]))

    @torch.no_grad()
    def transcribe(self, q, max_len=4):
        """Greedy decoding until the end token."""
        tin, out = torch.full((q.shape[0], 1), BOS_T), [[] for _ in range(q.shape[0])]
        done = torch.zeros(q.shape[0], dtype=torch.bool)
        for _ in range(max_len):
            nxt = self.logits(q, tin)[:, -1].argmax(-1)
            for b in range(q.shape[0]):
                if not done[b]:
                    done[b] = bool(nxt[b] > 9)
                    if not done[b]:
                        out[b].append(int(nxt[b]))
            tin = torch.cat([tin, nxt[:, None]], 1)
        return out


class CAT(nn.Module):
    def __init__(self, d=64, layers=(2, 2, 2, 2)):
        super().__init__()
        p0 = PATCH[0]
        enc = [(p0, d // 2), (d, d // 2), (d, d // 2), (d, d)]  # (in, out) of each stage
        dec = [(d, d), (d // 2, d), (d // 2, d), (d // 2, 2 * (2 * p0 + 1))]
        self.enc = nn.ModuleList(Stage(i, d, n, o) for (i, o), n in zip(enc, layers, strict=True))
        self.dec = nn.ModuleList(Stage(i, d, n, o) for (i, o), n in zip(dec, layers[::-1], strict=True))
        self.rvq = RVQ(d)
        self.sem = SemanticLM(d)

    def encode_z(self, x):
        h = x.view(x.shape[0], -1, PATCH[0])  # (B, 128, 80) for a 1.28 s clip
        for s, st in enumerate(self.enc):
            if s:
                h = h.reshape(h.shape[0], h.shape[1] // 2, -1)  # patchify: merge 2 neighbouring positions
            h = st(h)
        return h  # (B, 16, 64): 12.5 vectors per second

    def decode_q(self, q):
        h = q
        for s, st in enumerate(self.dec):
            h = st(h)
            if s < len(self.dec) - 1:
                h = h.reshape(h.shape[0], h.shape[1] * 2, -1)  # unpatchify: split 1 position into 2
        p0 = PATCH[0]
        n = 2 * p0 + 1  # the 161 frequencies of a 320-sample window
        mag = h[..., :n].clamp(max=math.log(100)).exp()
        frames = torch.fft.irfft(torch.polar(mag, h[..., n:]), 4 * p0) * torch.hann_window(4 * p0) / 1.5
        return overlap_add(frames, p0)  # one 40 ms window every 10 ms

    @torch.no_grad()
    def encode(self, x, k=NQ):
        return self.rvq.encode(self.encode_z(x), k)

    @torch.no_grad()
    def decode(self, codes):
        return self.decode_q(self.rvq.decode(codes))

Training this tokenizer takes more than an hour on 4 CPU cores, too long for a notebook. The course's script
`reel/codec/sweep.py` trained it (10,000 steps, batches of 32 clips of 1.28 seconds, AdamW, η = 3 · 10⁻³ chosen from
{10⁻³, 3 · 10⁻³, 10⁻²} on the validation set) with the paper's loss and weights:
15 · ℒ_mel + 0.25 · ℒ_cmt + 1 · ℒ_code + 20 · ℒ_sem, and quantizer dropout on every example. We load seed 0, stored
as int8 (one scale factor per row); every number the script reports is computed on exactly these int8 weights.

In [ ]:
R = fetch_json("codec-results.json", "53c34e984a6054d6135be5afd7c9128d")
W = fetch_json("codec-weights.json", "6f4b0e5508a869f770f628dd0468e742")


def unpack(p):
    """Unpack the weights: int8 × a float16 scale per row, or plain float16."""
    sd = {}
    for k, v in p.items():
        if "q" in v:
            q = np.frombuffer(base64.b64decode(v["q"]), np.int8).astype(np.float32).reshape(v["shape"][0], -1)
            s = np.frombuffer(base64.b64decode(v["s"]), "<f2").astype(np.float32)
            sd[k] = torch.tensor((q * s[:, None]).reshape(v["shape"]))
        else:
            raw = np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float32)
            sd[k] = torch.tensor(raw.reshape(v["shape"]))
    return sd


codec = CAT()
codec.load_state_dict(unpack(W["codec"]))
codec.eval()
n_codec = sum(p.numel() for n, p in codec.named_parameters() if not n.startswith("sem."))
n_sem = sum(p.numel() for p in codec.sem.parameters())
print(
    f"miniature CAT: {num(n_codec, 0)} parameters, plus {num(n_sem, 0)} in the semantic model; the paper: 1.6 billion"
)
assert n_codec == R["codec"]["ref"][0]["params"] and W["cfg"]["steps"] == R["steps"] == 10_000
assert R["eta"]["chosen"] == 3e-3 and R["eta"]["grid"] == [1e-3, 3e-3, 1e-2]

x1 = pad_clips([EX])
h = x1.view(1, -1, PATCH[0])
print(f"{'1.28 s clip cut into 80-sample patches':40s} {tuple(h.shape)}")
with torch.no_grad():
    for s, st in enumerate(codec.enc):
        if s:
            h = h.reshape(1, h.shape[1] // 2, -1)
            print(f"{'  patchify (merge 2 positions)':40s} {tuple(h.shape)}")
        h = st(h)
        print(f"{f'stage {s + 1} (2 blocks, width 64)':40s} {tuple(h.shape)}")
    codes_ex = codec.rvq.encode(h, NQ)
print(f"{'RVQ: 32 tokens for each 80 ms frame':40s} {tuple(codes_ex.shape)}")
print("the 32 tokens of frame 5:", codes_ex[0, 4].tolist())

**Causal, so it can stream.** Because every block only looks at the past, the vectors of the first T frames must be
the same whether we feed in the first T · 80 ms or the whole 1.28 seconds: the encoder doesn't have to wait for the
utterance to end. On the decoding side, the output up to 30 ms before the end of the last frame must not change
either; the remaining 30 ms wait for the next frame (the price of the spectral output described above).

In [ ]:
with torch.no_grad():
    z_full, codes_full = codec.encode_z(x1), codec.encode(x1)
    y_full = codec.decode(codes_full)
    for T in (2, 3, 4, 5):
        assert torch.allclose(codec.encode_z(x1[:, : T * HOP]), z_full[:, :T], atol=1e-5)
        y_T = codec.decode(codes_full[:, :T])
        done = T * HOP - 3 * PATCH[0]
        assert torch.allclose(y_T[:, :done], y_full[:, :done], atol=1e-5)
        tail = float((y_T[:, done:] - y_full[:, done : T * HOP]).abs().max())
        print(f"first {T} frames: encoder vectors identical to the full 1.28 s; audio identical up to "
              f"{num(1000 * done / SR, 0)} ms, the last 30 ms differ by up to {num(tail, 3)}")  # fmt: skip

### Hearing a digit through 1 … 32 layers

The same "seven" from section 2 is encoded once into 16 × 32 tokens, then decoded from only the first K layers.
Listen from 1 layer (125 bits/s) up to 32 layers (4,000 bits/s), and notice when you can tell the digit and when you
can tell the speaker's voice.

In [ ]:
with torch.no_grad():
    recon = {k: codec.decode(codes_full[..., :k])[0, : len(x_ex)].numpy() for k in (1, 2, 4, 8, 16, 32)}
play(x_ex, "original (128,000 bits/s)")
for k, y in recon.items():
    play(y, f"{nlayers(k)} = {num(FRAME_RATE * k * BITS, 0)} bits/s")

fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
spectrogram(axes[0], x_ex, "original")
for ax, k in zip(axes[1:], (1, 4, 32), strict=True):
    spectrogram(ax, recon[k], f"{nlayers(k)} ({num(FRAME_RATE * k * BITS, 0)} bits/s)")
for ax in axes[1:]:
    ax.set_ylabel("")
fmt_axes(fig)
plt.tight_layout()
plt.show()

### Measuring with two "judges"

The paper measures with off-the-shelf models: the word error rate (WER) of a speech recognizer, and voice similarity
(SIM), the cosine between a speaker-verification model's embeddings of the original and of the rebuild. We build two
small stand-in judges, trained on the *original* training recordings (not on anything that went through the codec):
one CNN guesses the digit, another guesses the speaker. The second CNN's 128-dimensional embedding gives SIM. Half of
their training batches are randomly degraded (noise, low-pass filtering, spectral tilt), so they "hear through"
distortion as a real model would, instead of falling apart over small differences. On the original test recordings
they score:

In [ ]:
class Judge(nn.Module):
    """1-D CNN over 40 log-mel bands; mean + max pooling over time → 128-d embedding → classes."""

    def __init__(self, classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(40, 128, 5, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
        )  # fmt: skip
        self.emb, self.head = nn.Linear(256, 128), nn.Linear(128, classes)

    def embed(self, x):
        h = self.conv((logmel(x) + 5) / 2)
        return F.relu(self.emb(torch.cat([h.mean(-1), h.amax(-1)], 1)))

    def forward(self, x):
        return self.head(self.embed(x))


jd, js = Judge(10), Judge(len(SPEAKERS))
jd.load_state_dict(unpack(W["judgeDigit"]))
js.load_state_dict(unpack(W["judgeSpk"]))
jd.eval(), js.eval()
x_test = pad_clips(TEST)
with torch.no_grad():
    acc_d = float((jd(x_test).argmax(-1).numpy() == digit[TEST]).mean())
    acc_s = float((js(x_test).argmax(-1).numpy() == spk[TEST]).mean())
print(f"digit judge: {pct(acc_d)} · voice judge: {pct(acc_s)} (300 original test recordings)")
assert abs(acc_d - R["judges"]["digit"]["test"]) <= 1 / 300 and abs(acc_s - R["judges"]["spk"]["test"]) <= 1 / 300

### The tokenizer on the test set

> **🤔 Predict first:** how many layers of tokens does a judge need to recognize the *digit*? How many to recognize
> the *voice*? Which one needs more bits?

The cell below runs the seed-0 tokenizer live on 60 test recordings (one per speaker per digit) at 1, 4 and 32 layers,
then compares the result with the script's numbers for those same recordings. The table after it covers all 300
recordings, averaged over 3 seeds. Besides the two judges, the tokenizer's own semantic model can "listen" too: it
reads the tokens and writes out the digit (the *ASR from tokens* column).

In [ ]:
PC = W["perClip"]
pos = {c: i for i, c in enumerate(PC["ids"])}
SUB = [c for c in TEST if idx[c] == 0]
x_sub = pad_clips(SUB)
f_sub = np.array([-(-n_samples[c] // HOP) for c in SUB])
mask = (torch.arange(WIN * HOP)[None] < torch.tensor(f_sub * HOP)[:, None]).float()
with torch.no_grad():
    c_sub = codec.encode(x_sub)
    for k in (1, 4, 32):
        y = codec.decode(c_sub[..., :k]) * mask
        live = {
            "mel": [float(mel_loss(y[i : i + 1, : f * HOP], x_sub[i : i + 1, : f * HOP])) for i, f in enumerate(f_sub)],
            "digit": (jd(y).argmax(-1).numpy() == digit[SUB]).astype(int),
            "spk": (js(y).argmax(-1).numpy() == spk[SUB]).astype(int),
            "sim": F.cosine_similarity(js.embed(y), js.embed(x_sub)).numpy(),
            "asr": [int(t == [d]) for t, d in zip(codec.sem.transcribe(codec.rvq.decode(c_sub[..., :k])), digit[SUB])],
        }
        ki = PC["k"].index(k)
        ref_ = {key: np.array([PC[key][ki][pos[c]] for c in SUB]) for key in live}
        same = {key: float(np.mean(np.isclose(live[key], ref_[key], atol=2e-3))) for key in live}
        print(f"{nlayers(k):>9s}: mel {num(np.mean(live['mel']), 3)} · digit {pct(np.mean(live['digit']))} · voice "
              f"{pct(np.mean(live['spk']))} · SIM {num(np.mean(live['sim']), 3)} · ASR from tokens "
              f"{pct(np.mean(live['asr']))}; matches the script on {pct(min(same.values()), 0)} of recordings")  # fmt: skip
        assert min(same.values()) >= 0.95


def seeds(name, split="test"):
    return {key: np.array([r[split][key] for r in R["codec"][name]]) for key in ("mel", "asr", "digit", "spk", "sim")}


ref = seeds("ref")
print("\n300 test recordings, mean of 3 seeds")
print(f"{'layers':>6s} | {'bits/s':>6s} | {'mel':>4s} | {'ASR from tokens':>15s} | {'digit':>6s} | {'voice':>6s} | SIM")
for i, k in enumerate(R["ks"]):
    print(f"{k:6d} | {num(FRAME_RATE * k * BITS, 0):>6s} | {num(ref['mel'][:, i].mean(), 2)} | "
          f"{pct(ref['asr'][:, i].mean()):>15s} | {pct(ref['digit'][:, i].mean()):>6s} | "
          f"{pct(ref['spk'][:, i].mean()):>6s} | {num(ref['sim'][:, i].mean(), 3)}")  # fmt: skip

Reading the table from the top:

- **Content is cheap, the voice is expensive.** With just 1 layer (125 bits/s), the judge already hears the right
  digit 81.9% of the time, but recognizes the voice only 72.0% of the time. The voice needs about 4 layers
  (500 bits/s) to reach 90.6%. That fits the count in section 4: the content needs only a few bits per second, and
  most of the remaining bits describe who is speaking.
- **The tokens know more than the rebuilt audio does.** The semantic model reads the 1-layer tokens directly and
  writes the right digit 92.4% of the time, while the judge listening to audio decoded from those same tokens is
  right only 81.9% of the time. The information is in the tokens; the small decoder doesn't rebuild all of it. (The
  judge learned on the original recordings, while the semantic model learned on these very tokens, so the comparison
  favours the semantic model.)
- **From about 8 layers on, more bits barely help:** the mel distance is 1.62 at 4 layers, 1.59 at 8 and still 1.59
  at 32, well short of what the judges get on the originals (97.7% digit, 96.3% voice). At this size the model is
  the bottleneck, not the number of bits. The spectrograms above show it: even at 32 layers only the lowest one or two
  harmonics stay sharp, and the higher ones blur into a broad band. Section 8 (C) tries a wider model.

Another way to see that the last layers carry little: count how many codes each layer actually uses. If a layer uses
all 1,024 codes equally often, each of its tokens is worth 10 bits; if it almost always picks a few favourite codes,
much less. The entropy of the code frequencies, in bits, is what an ideal compressor needs per token of that layer
(ignoring the relations between layers). The script counted the codes of the seed-0 tokenizer over every frame of
the training set.

In [ ]:
U = PC["usage"]
eff = np.log2(U["perplexity"])  # perplexity = 2^entropy, so its log₂ is the entropy in bits
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, NQ + 1), eff, color=BLUE, label="entropy of the code frequencies")
ax.axhline(BITS, color=GREY, ls="--", lw=1, label="at most 10 bits: all 1,024 codes used equally")
ax.set(xlabel="RVQ layer", ylabel="bits per token", title=f"{num(U['frames'], 0)} frames of the training set, seed 0")
ax.set_ylim(0, 12.5)
ax.legend(fontsize=8, loc="upper right", ncol=2, frameon=False)
fmt_axes(fig)
plt.tight_layout()
plt.show()
print(f"layer 1 uses {num(U['distinct'][0], 0)} different codes, layer 8 uses {num(U['distinct'][7], 0)}, layer 32 "
      f"only {num(U['distinct'][-1], 0)}")  # fmt: skip
print(f"in total: {num(eff.sum(), 1)} bits per frame instead of {num(NQ * BITS, 0)}, that is about "
      f"{num(eff.sum() * FRAME_RATE, 0)} bits/s instead of 4,000")  # fmt: skip
assert eff[-4:].max() < eff[1:16].min() and U["frames"] == sum(-(-n_samples[c] // HOP) for c in TRAIN)

Layers 2 to about 17 use almost the whole codebook (9.2–9.8 bits per token), while the last layers use a few hundred
codes, down to 18. Part of the reason is probably the quantizer dropout from section 6: layer k only learns on the
examples that keep at least k layers, that is (33 − k)/32 of them, so layer 32 learns on only 1/32 of the examples.
Layer 1 doesn't use the whole codebook either (8.1 bits). So the nominal rate is 4,000 bits/s, but a good compressor
would only need about 3,300.

## 8 · Testing the claims

Each configuration below is the tokenizer from section 7 with exactly *one* change, the same 10,000 steps, the same
η and 3 seeds. Each line is the mean of the 3 seeds, and the shaded band runs from the lowest seed to the highest.
Every measurement is on the 300 test recordings.

In [ ]:
BPS = FRAME_RATE * np.array(R["ks"]) * BITS
STY = [(BLUE, "o", "-"), (ORANGE, "s", "--"), (GREEN, "^", ":"), (GREY, "D", "-.")]
YL = {"mel": "mel distance (lower is better)", "digit": "digit judge correct (%)", "spk": "voice judge correct (%)",
      "sim": "SIM", "asr": "ASR from tokens (%)"}  # fmt: skip
SCALE = {"mel": 1, "sim": 1, "digit": 100, "spk": 100, "asr": 100}


def band(ax, x, a, sty, label):
    color, marker, ls = sty
    ax.plot(x, a.mean(0), marker=marker, ls=ls, color=color, label=label, ms=4)
    ax.fill_between(x, a.min(0), a.max(0), color=color, alpha=0.15, lw=0)


def compare(names, labels, keys=("mel", "spk"), title=""):
    """Measurements against the bitrate for a few configurations."""
    fig, axes = plt.subplots(1, len(keys), figsize=(5.2 * len(keys), 3.4))
    for ax, key in zip(np.atleast_1d(axes), keys, strict=True):
        for sty, name, lab in zip(STY, names, labels, strict=False):
            band(ax, BPS, SCALE[key] * seeds(name)[key], sty, lab)
        ax.set(xscale="log", xlabel="bits per second", ylabel=YL[key])
    np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.suptitle(title)
    fmt_axes(fig)
    plt.tight_layout()
    plt.show()


def table(names, labels, ks=(1, 2, 4, 8, 32), keys=("mel", "digit", "spk", "sim")):
    """Mean of 3 seeds (lowest–highest) at a few bitrates."""
    w = max(len(lab) for lab in labels) + 2
    for key in keys:
        print(f"{YL[key]}:")
        print(" " * (w + 3) + "".join(f"{nlayers(k):>22s}" for k in ks))
        for name, lab in zip(names, labels, strict=True):
            a = SCALE[key] * seeds(name)[key]
            cells = []
            for k in ks:
                i = R["ks"].index(k)
                d = 1 if key in ("digit", "spk", "asr") else (3 if key == "sim" else 2)
                cells.append(f"{num(a[:, i].mean(), d)} ({num(a[:, i].min(), d)}–{num(a[:, i].max(), d)})")
            print(f"   {lab:{w}s}" + "".join(f"{c:>22s}" for c in cells))


def curves(names, labels, key, k, title=""):
    """Validation measurements during training, at k layers."""
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for sty, name, lab in zip(STY, names, labels, strict=False):
        runs = R["codec"][name]
        steps = [c["step"] for c in runs[0]["curve"]]
        ki = (2, 8, 32).index(k)
        a = SCALE[key] * np.array([[c[key][ki] for c in r["curve"]] for r in runs])
        band(ax, steps, a, sty, lab)
    ax.set(xlabel="training step", ylabel=YL[key] + f", {nlayers(k)}", title=title)
    ax.legend(fontsize=8)
    fmt_axes(fig)
    plt.tight_layout()
    plt.show()

### A · Quantizer dropout: on how many examples?

The paper applies quantizer dropout to every example (rate 1) without comparing other rates. We try 0 (always all
32 layers), ½ (as in DAC's release) and 1: the experiment of section 6, on the real tokenizer.

> **🤔 Predict first:** going by the small problem in section 6, which configuration wins at 1 layer, and which at
> 32 layers?

In [ ]:
QD = (["qdrop0", "qdrop05", "ref"], ["dropout 0", "dropout ½", "dropout 1 (paper)"])
compare(*QD, keys=("mel", "spk"), title="A · quantizer dropout rate")
table(*QD)

**A: holds, with a trade-off the paper doesn't mention.** Without quantizer dropout the tokenizer is nearly useless
when it is cut short: at 1 layer the mel distance is 4.52 against 1.98, and the judge hears the right digit only
28.8% of the time against 81.9%. But at all 32 layers it is the best: mel 1.51 against 1.59 (the 3-seed bands don't
overlap), digit 93.8% against 90.3%. This is the small problem from section 6 again: a model that only learns with
all 32 layers spreads the work over all 32, so the first layer is never taught to stand alone. DAC's rate ½ sits in
between: close to rate 1 on mel and digit at every bitrate, but clearly worse on the voice at 1 layer (58.2%
against 72.0%), and at 32 layers different from rate 1 only within the seed range. For a tokenizer that a language
model will read at any number of layers, the paper's rate 1 is a sensible choice.

### B · End-to-end or "partial" training

Many earlier tokenizers were trained in stages: learn the encoder and the quantizer, freeze them, then keep training
only the decoder. The paper (Figure 4) compares the two approaches and finds that "partial" training stalls early. We
run the same experiment: from the exact midpoint (step 5,000) of the reference tokenizer, one branch keeps training
everything, and the other freezes the encoder, the RVQ and the semantic model. Both branches see exactly the same
batches.

In [ ]:
PARTIAL = (["ref", "partial"], ["end-to-end (paper)", "partial (frozen)"])
curves(*PARTIAL, "mel", 8, "B · mel distance on the validation set")
curves(*PARTIAL, "spk", 8, "B · voice judge on the validation set")
table(*PARTIAL)

**B: holds.** The two branches start from the same point at step 5,000. The partial branch still improves, but much
more slowly, and the gap widens at every measurement: at step 10,000, on the test set, the mel distance at 8 layers
is 1.78 against 1.59, and the voice judge is right 83.0% of the time against 91.7%; the 3-seed bands don't overlap at
any bitrate. Freezing the encoder and the quantizer also freezes the tokens at their step-5,000 quality, and the
decoder can only make the best of those tokens. End-to-end training can still improve the tokens themselves. We only
trained 5,000 more steps, so we can't tell whether the partial branch levels off completely, as in the paper's
Figure 4.

### C · Bigger models and bitrate

The paper (Figure 5) changes the width of every block (from 319 million to 1.17 billion parameters) and finds that
bigger models are better, especially at high bitrates, while at low bitrates the bits are the bottleneck. We vary the
width: 32 / 64 / 96.

In [ ]:
SIZE = (["d32", "ref", "d96"], ["width 32", "width 64", "width 96"])
for n, lab in zip(*SIZE, strict=True):
    print(f"{lab}: {num(R['codec'][n][0]['params'], 0)} parameters")
compare(*SIZE, keys=("mel", "spk"), title="C · model width")
table(*SIZE)

**C: holds, but the step from width 64 to 96 is within the seed range.** At 32 layers the mel distance is 1.80 /
1.59 / 1.54 for widths 32 / 64 / 96; at 1 layer it is 2.12 / 1.98 / 1.96. Bigger models gain more at high bitrates:
from width 32 to 64, mel drops by 0.22 at 32 layers but only by 0.13 at 1 layer; from 64 to 96, it drops by 0.05 at
32 layers but only by 0.02 at 1 layer. At low bitrates the three lines are close: the bits are the bottleneck, as in
Figure 5. Two caveats: the 3-seed bands of widths 64 and 96 overlap, and every size uses the η chosen for width 64
(the paper, too, keeps its optimizer settings when it changes the size). Width 32 is clearly worse on the voice
(71.8% at 32 layers against 92.2%).

### D · Larger batches, same number of steps

The paper (Figure 6) keeps the number of steps fixed and raises the batch size, and finds larger batches better at
*every* point of training. We try batches of 8, 16 and 32 clips of 1.28 seconds.

In [ ]:
BATCH = (["bs8", "bs16", "ref"], ["batch 8", "batch 16", "batch 32"])
curves(*BATCH, "mel", 8, "D · mel distance on the validation set")
table(*BATCH)

**D: holds, from step 2,000 on.** At the first measurement (step 1,000), batch 16 is still slightly ahead of batch 32
on the validation set; from step 2,000 on, batch 32 beats batch 16 and batch 16 beats batch 8 at every measurement.
At step 10,000, on the test set at 32 layers: mel 2.03 / 1.80 / 1.59 and voice judge 44.0% / 76.7% / 92.2% for
batches of 8 / 16 / 32. Like the paper's experiment, this comparison fixes the number of steps, so larger batches also
see more data and cost more compute: batch 32 sees 320,000 clips, batch 8 only 80,000. It shows that a large batch is
the better use of a fixed number of steps, but it doesn't isolate the effect of the batch size itself.

### E · Removing the semantic loss

The paper attaches an LLM with 0.5 billion parameters so that the tokens "carry meaning", but never tries removing
it. We remove the semantic model (λ_sem = 0) and keep everything else.

> **🤔 Predict first:** without the semantic loss, will the mel distance get better or worse? And the digit judge?

In [ ]:
SEM = (["sem0", "ref"], ["no semantic loss", "λ_sem = 20 (paper)"])
compare(*SEM, keys=("mel", "digit"), title="E · semantic loss")
table(*SEM)

**E: depends.** Without the semantic loss the mel distance is better at *every* bitrate (1.48 against 1.59 at 32
layers; 1.88 against 1.98 at 1 layer; the 3-seed bands don't overlap at any bitrate). From 4 layers on, the digit
judge is also slightly better (92.9% against 90.3% at 32 layers), but only within the seed range. At 1 layer it is
the other way round: the judge hears the right digit only 71.9% of the time against 81.9%, and the 3-seed bands don't
overlap. The semantic loss pushes "what is said" into the first layer of tokens, at the cost of some sound quality.
That is worth it when a language model reads only the first few layers, and not when you need the best sound at all
32 layers. We only tried the paper's λ_sem = 20; a smaller λ might keep the benefit at a lower cost.

### Spectral or sample output

Last, the departure from the paper described in section 7: everything is the same, but the decoder directly outputs
the 80 waveform samples of each 10 ms position, as the paper's CAT does.

In [ ]:
HEAD = (["patch", "ref"], ["sample output (paper)", "spectrum + iSTFT (here)"])
compare(*HEAD, keys=("mel", "spk"), title="decoder output")
table(*HEAD)

At this size the spectral output is better: mel 1.59 against 1.68 at 32 layers (the 3-seed bands don't overlap), and
the voice judge 92.2% against 87.1%. The digit is about the same (90.3% against 90.8%). The gap is smaller than the
3,000-step pilot runs suggested: given enough steps, the paper's approach learns too. The paper doesn't need this
trick at 1.6 billion parameters; here it buys quality for 30 ms of delay.

## 9 · A miniature CAT-TTS and Progressive Sequence Dropout

From CAT's tokens the paper builds CAT-TTS: a **fully autoregressive** model that reads text aloud. Each frame has 32
tokens, so a 10-second utterance is 125 × 32 = 4,000 tokens. Instead of flattening them, CAT-TTS splits the work
between two Transformers (the layout of RQ-Transformer, Lee et al. 2022, which Moshi calls Temporal + Depth):

- The **Temporal Transformer** runs over time, one position per frame. The input for frame t is the *sum* of the
  embeddings of its tokens, with a separate embedding table for each layer (Eq. 8): ẽₜ = Σₖ Embₖ(qₜ,ₖ).
- The smaller **Depth Transformer** runs *inside* one frame: from the Temporal Transformer's state it predicts the
  layer-1 token, then the layer-2 token given layer 1, and so on up to layer 32.

The paper's TTS model also takes an audio prompt: [the prompt's transcript, the text to read, the prompt's tokens] →
the tokens of the text to read, *in the prompt's voice*. Ours keeps that setup, with two digits as the text: the
prompt's digit and the digit to read.

**Progressive Sequence Dropout (PSD)**, the paper's second contribution: with probability p, a training example
keeps only its first K̂ layers (K̂ drawn uniformly), both in the input sum and in the loss (Eq. 6–9); the other
examples keep all 32 layers. That way the number of layers K can be chosen at generation time: fewer layers are
faster and use fewer bits, and the model has seen exactly that situation in training. The paper writes
K̂ ∈ {1, …, 31}; then at p = 1 layer 32 is never trained, so we draw K̂ ∈ {1, …, 32}.

In [ ]:
BOA, EOS_C = 10, CODES  # text: digits 0–9 and "begin audio"; layer 1 has one extra choice, "end of utterance"


class TTS(nn.Module):
    """[prompt digit, digit to read, BOA], then the audio frames. The Temporal Transformer runs over time on the
    summed embeddings of the first K̂ layers (Eq. 8); from its state at frame t, the Depth Transformer predicts the 32
    tokens of frame t + 1 one by one."""

    def __init__(self, nq=NQ, d=128, layers=4, dd=64, depth_layers=2):
        super().__init__()
        self.text = nn.Embedding(11, d)
        self.emb = nn.ModuleList(nn.Embedding(CODES, d) for _ in range(nq))  # Embₖ, one table per layer
        self.temporal = nn.ModuleList(Block(d, ls=1.0) for _ in range(layers))
        self.tnorm = nn.LayerNorm(d)
        self.ctx = nn.Linear(d, dd)
        self.prev = nn.ModuleList(nn.Embedding(CODES, dd) for _ in range(nq - 1))
        self.depth = nn.ModuleList(Block(dd, ls=1.0) for _ in range(depth_layers))
        self.dnorm = nn.LayerNorm(dd)
        self.heads = nn.ModuleList(nn.Linear(dd, CODES + (k == 0)) for k in range(nq))
        self.nq, self.dh, self.ddh = nq, d // 4, dd // 4

    def frame_in(self, codes, k_hat):
        """Eq. 8: ẽₜ = Σ_{k ≤ K̂} Embₖ(q_{t,k}); each example has its own K̂."""
        out = 0
        for k in range(int(k_hat.max())):
            out = out + self.emb[k](codes[..., k].clamp_min(0)) * (k < k_hat).float()[:, None, None]
        return out

    def temporal_states(self, text, codes, k_hat):
        h = torch.cat([self.text(text), self.frame_in(codes, k_hat)], 1)
        cs = rope(h.shape[1], self.dh)
        for b in self.temporal:
            h = b(h, cs)
        return self.tnorm(h)

    def depth_logits(self, ctx, prev, K):
        """Position 1 sees only the temporal state; position k also sees this frame's token of layer k − 1."""
        h = torch.stack([self.ctx(ctx)] + [self.ctx(ctx) + self.prev[k](prev[:, k]) for k in range(K - 1)], 1)
        cs = rope(K, self.ddh)
        for b in self.depth:
            h = b(h, cs)
        h = self.dnorm(h)
        return [self.heads[k](h[:, k]) for k in range(K)]

    def loss(self, text, codes, n_frames, k_hat):
        """Eq. 9: −Σₜ Σ_{k ≤ K̂} log p(q_{t,k} | text, earlier frames, earlier layers of frame t), divided by the number
        of frames. codes (B, T, 32): the prompt's frames followed by the target's (−1 is padding); after the last frame
        comes "end of utterance"."""
        h = self.temporal_states(text, codes[:, :-1], k_hat)
        ctx = h[:, 2:]  # the state at BOA predicts frame 1, the state at frame t predicts frame t + 1
        t_idx = torch.arange(ctx.shape[1])[None]
        valid, is_eos = t_idx <= n_frames[:, None], t_idx == n_frames[:, None]
        target = codes[:, : ctx.shape[1]].clone()
        target[..., 0] = torch.where(is_eos, torch.tensor(EOS_C), target[..., 0])
        sel = valid.reshape(-1)
        c, tg = ctx.reshape(-1, ctx.shape[-1])[sel], target.reshape(-1, self.nq)[sel]
        kk = k_hat[:, None].expand(-1, ctx.shape[1]).reshape(-1)[sel]
        eos = is_eos.reshape(-1)[sel]
        K = int(kk.max())
        logits = self.depth_logits(c, tg[:, : K - 1].clamp(0, CODES - 1), K)
        total, count = 0.0, 0
        for k in range(K):
            m = (k < kk) & (~eos | (k == 0))  # only the kept layers; the end-of-utterance frame has only layer 1
            total = total + F.cross_entropy(logits[k][m], tg[m, k], reduction="sum")
            count += int(m.sum())
        return total / sel.sum(), count


tts = TTS()
print(f"miniature CAT-TTS: {num(sum(p.numel() for p in tts.parameters()), 0)} parameters (untrained, only to show the "
      "shapes)")  # fmt: skip

The cell below takes 8 pairs of training recordings (each pair from one speaker), encodes them with the CAT above,
then draws K̂ for each pair as PSD does, with p = 0, ½ and 1. The figure shows which layers are kept (blue) in the
input sum and in the loss. An untrained model gives all 1,024 codes about the same probability, so its loss is about
ln 1,024 ≈ 6.93 nats per kept token: a check that the loss counts only the kept layers.

In [ ]:
pairs = [(TRAIN[i], TRAIN[i + 1]) for i in range(0, 160, 20)]
assert all(spk[a] == spk[b] and a != b for a, b in pairs)
clips = sorted({c for pr in pairs for c in pr})
with torch.no_grad():
    enc = codec.encode(pad_clips(clips))
tok = {c: enc[i, : -(-n_samples[c] // HOP)] for i, c in enumerate(clips)}
seqs = [torch.cat([tok[a], tok[b]]) for a, b in pairs]
n_fr = torch.tensor([len(q) for q in seqs])
batch_codes = torch.full((len(pairs), int(n_fr.max()) + 1, NQ), -1, dtype=torch.long)
for i, q in enumerate(seqs):
    batch_codes[i, : len(q)] = q
batch_text = torch.tensor([[digit[a], digit[b], BOA] for a, b in pairs])

rng_psd = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8), sharey=True)
for ax, p in zip(axes, (0.0, 0.5, 1.0), strict=True):
    z = rng_psd.random(len(pairs)) < p  # Eq. 6
    k_hat = torch.tensor(np.where(z, rng_psd.integers(1, NQ + 1, len(pairs)), NQ))  # Eq. 7
    with torch.no_grad():
        loss, count = tts.loss(batch_text, batch_codes, n_fr, k_hat)
    per_token = float(loss) * int(n_fr.sum() + len(pairs)) / count
    print(
        f"p = {num(p, 1)}: K̂ = {k_hat.tolist()}; {num(count, 0)} tokens in the loss, {num(per_token, 2)} nats per token"
    )
    assert abs(per_token - math.log(CODES)) < 0.7
    ax.imshow(np.arange(NQ)[None] < k_hat.numpy()[:, None], cmap=plt.matplotlib.colors.ListedColormap(["#e6e9ef", BLUE]),
              aspect="auto", extent=[0.5, NQ + 0.5, len(pairs) + 0.5, 0.5])  # fmt: skip
    ax.set(title=f"p = {num(p, 1)}", xlabel="RVQ layer")
axes[0].set_ylabel("training example")
fmt_axes(fig)
plt.tight_layout()
plt.show()

### Results: 4 values of p × 3 seeds

The script trained the miniature CAT-TTS on the tokens of the seed-0 tokenizer with p = 0, ¼, ½ and 1, 3 seeds each,
for 4,000 steps with batches of 64 pairs. One trick was needed at this size: each recording is encoded 7 more times,
each time shifted by another 10 ms before encoding, so the frame boundaries fall in other places and the tokens
change. Without it, the model memorizes the tokens of the 2,397 training recordings. At test time each test
recording serves as the prompt, and the model reads a *different* digit (chosen in advance and fixed) at K = 1, 2, 4,
8, 16 and 32 layers, sampling at temperature 0.8 with top-k 50 (chosen in advance, not tuned). The digit judge
stands in for WER; the voice judge and SIM (against the prompt) stand in for the paper's SIM.

Two reference points. The **"real recording"** is the prompt's own speaker reading the target digit (another test
recording), passed through the tokenizer at the same K: the best a perfect TTS could reach. And SIM between two real
recordings of the *same* speaker, against two recordings by *different* speakers.

> **🤔 Predict first:** at 32 layers, are the p = 0 and p = 1 models very different? And at 1 layer?

In [ ]:
TT, BASE = R["tts"], R["baselines"]
TKS = np.array(BASE["oracle"]["k"])
PS_ = ["0", "0.25", "0.5", "1"]
PLAB = {"0": "p = 0 (no PSD)", "0.25": "p = ¼", "0.5": "p = ½", "1": "p = 1"}
assert TT["1"][0]["params"] == sum(p.numel() for p in tts.parameters()) and R["ttsSteps"] == 4000

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, key in zip(axes, ("digit", "spk", "sim"), strict=True):
    sc = 1 if key == "sim" else 100
    for sty, p in zip(STY, PS_, strict=True):
        band(ax, TKS, sc * np.array([r["test"][key] for r in TT[p]]), sty, PLAB[p])
    ax.plot(TKS, sc * np.array(BASE["oracle"][key]), color="k", lw=1, label="real recording through the tokenizer")
    if key == "sim":
        for kind, lab, ls in (
            ("same", "two real recordings, same speaker", "--"),
            ("other", "different speakers", ":"),
        ):
            ax.axhline(BASE["real"][kind]["sim"], color=GREY, ls=ls, lw=1, label=lab)
    else:
        ax.axhline(100 / (10 if key == "digit" else len(SPEAKERS)), color=GREY, ls=":", lw=1, label="chance")
    ax.set(
        xscale="log",
        xlabel="layers K at generation",
        ylabel={"digit": YL["digit"], "spk": YL["spk"], "sim": "SIM"}[key],
    )
    ax.xaxis.set_major_locator(FixedLocator(list(TKS)))
h0, l0 = axes[0].get_legend_handles_labels()
h2, l2 = axes[2].get_legend_handles_labels()
fig.legend(h0 + h2[len(l0) - 1 :], l0 + l2[len(l0) - 1 :], loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.suptitle("F · miniature CAT-TTS: reading another digit in the prompt's voice")
fmt_axes(fig)
for ax in axes:
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _p: f"{v:g}"))
plt.tight_layout(rect=(0, 0.16, 1, 1))
plt.show()

TNAME = {"digit": "digit judge correct", "spk": "voice judge correct", "sim": "SIM against the prompt",
         "no_end": "utterance never ends"}  # fmt: skip
for key, name in TNAME.items():
    f = (lambda v: num(v, 3)) if key == "sim" else pct
    print(f"\n{name}, mean of 3 seeds (lowest–highest)")
    print(" K | " + " | ".join(f"{PLAB[p]:>21s}" for p in PS_) + (" | real recording" if key != "no_end" else ""))
    for i, k in enumerate(TKS):
        c = [np.array([r["test"][key][i] for r in TT[p]]) for p in PS_]
        row = " | ".join(f"{f(x.mean())} ({f(x.min())}–{f(x.max())})".rjust(21) for x in c)
        print(f"{k:2d} | {row}" + (f" | {f(BASE['oracle'][key][i])}" if key != "no_end" else ""))
print(f"\nSIM between two real recordings: same speaker {num(BASE['real']['same']['sim'], 3)}, "
      f"different speakers {num(BASE['real']['other']['sim'], 3)}")  # fmt: skip

**F: holds for intelligibility; as for the voice, every model copies it only partly.**

- **Without PSD, generating fewer layers makes the model misread.** At 1 layer the p = 0 model reads the right digit
  69.8% of the time, and the models with PSD 80.1–83.1% (the 3-seed bands don't overlap). At 1 layer, on average
  13.4% of the p = 0 model's utterances never end, but that number varies hugely between seeds (0.3–27.3%); with PSD
  it is only 0.8–1.3%. The p = 0 model has only ever seen the summed embeddings of all 32 layers as input; the sum of
  a single layer is something it has never met.
- **With PSD the lines are almost flat.** Reading with 1 layer is as good as with 32, and close to the real recording
  through the tokenizer at the same 1 layer (83.1% with p = 1, against 84.3%). At 1 layer the Depth Transformer runs
  1 step per frame instead of 32.
- **At 32 layers the gap narrows:** 74.9% with p = 0, against 77.1–81.0% with PSD; only p = 1 clears the seed band
  of p = 0. At every p, generating all 32 layers is no better than generating 8–16, and still far from the real
  recording (91.7%).
- **The voice:** with PSD, the voice judge recognizes the prompt's speaker 35–41% of the time at every K. Chance is
  16.7%, and the real recording through the tokenizer gets 66–92%. SIM of 0.86–0.88 sits between two real recordings
  of the same speaker (0.973) and of different speakers (0.711). The p = 0 model is worse at 1 layer (21.4%), and at
  32 layers on a par with the PSD models (SIM 0.873 against 0.861–0.869, within the seed range). The paper finds that
  PSD preserves both WER and SIM. Here PSD preserves intelligibility, while copying the voice is weak at every p,
  even though all 6 speakers are in the training data.

The ranking among p = ¼, ½ and 1 is mostly within the seed range: at this size, cutting just *some* of the examples
is enough.

### Listening to CAT-TTS

The data file holds the tokens of a few utterances the model generated (seed 0). The cell below decodes them with
the tokenizer from section 7, judges them again with both judges and compares the result with the script's. Each
prompt is speaker s reading digit s, and the model has to go on and read a different digit in that voice.

In [ ]:
SH = W["show"]
shown = {}
for p in ("0", "1"):
    s = SH[p]
    prompt_x = pad_clips(s["ids"])
    for ki, k in enumerate(s["k"]):
        n = [len(c) for c in s["codes"][ki]]
        padded = torch.zeros(len(n), WIN, k, dtype=torch.long)
        for j, c in enumerate(s["codes"][ki]):
            if n[j]:
                padded[j, : n[j]] = torch.tensor(c)
        with torch.no_grad():
            y = codec.decode(padded) * (torch.arange(WIN * HOP)[None] < torch.tensor(n)[:, None] * HOP).float()
            ok_d = (jd(y).argmax(-1).numpy() == np.array(s["target"])).astype(int)
            ok_s = (js(y).argmax(-1).numpy() == spk[s["ids"]]).astype(int)
            sim_ = F.cosine_similarity(js.embed(y), js.embed(prompt_x)).numpy()
        assert (ok_d != s["digit"][ki]).sum() <= 1 and (ok_s != s["spk"][ki]).sum() <= 1
        assert np.allclose(sim_, s["sim"][ki], atol=2e-3)
        shown[p, k] = (y, n)
        print(
            f"{PLAB[p]:15s} {nlayers(k):>9s}: right digit {ok_d.sum()}/6, right voice {ok_s.sum()}/6, "
            f"SIM {num(sim_.mean(), 3)}"
        )

for j in (0, 4):
    c = SH["1"]["ids"][j]
    play(waves[c], f'prompt: "{SPEAKERS[spk[c]]}" reads {digit[c]}; the model has to read {SH["1"]["target"][j]}')
    for p, k in (("0", 32), ("1", 32), ("0", 2), ("1", 2)):
        y, n = shown[p, k]
        play(y[j, : n[j] * HOP].numpy(), f"   {PLAB[p]}, {nlayers(k)}")

## 10 · Try it yourself

### What each layer carries

RVQ adds up the contribution of each layer. The cell below decodes the "seven" from layer 1 alone, from layers 2–32
*without* layer 1, and from all 32 layers. Listen, then try dropping other layers.

In [ ]:
with torch.no_grad():
    parts = [lay.outp(lay.book[codes_full[..., i]]) for i, lay in enumerate(codec.rvq.layers)]
    for name, q in (
        ("layer 1 only", parts[0]),
        ("layers 2–32, no layer 1", sum(parts[1:])),
        ("all 32 layers", sum(parts)),
    ):
        y = codec.decode_q(q)[:, : len(x_ex)]
        print(f"{name:24s} mel loss {num(mel_loss(y, x0), 2)}")
        play(y[0], name)

### Another recording

Change the speaker, the digit and the number of layers, then run the cell again.

In [ ]:
SPEAKER, DIGIT, N_LAYERS = "jackson", 3, 2
c = TEST[(spk[TEST] == SPEAKERS.index(SPEAKER)) & (digit[TEST] == DIGIT)][0]
with torch.no_grad():
    y = codec.decode(codec.encode(pad_clips([c]), N_LAYERS))[0, : len(waves[c])]
    guess_d = int(jd(y[None]).argmax())
    guess_s = SPEAKERS[int(js(y[None]).argmax())]
play(waves[c], f'original: "{SPEAKER}" reads {DIGIT}')
play(
    y,
    f"{nlayers(N_LAYERS)} ({num(FRAME_RATE * N_LAYERS * BITS, 0)} bits/s): the judges hear digit {guess_d}, "
    f"voice {guess_s}",
)

### The mel loss with fewer windows

The cell below recomputes the mel loss of the five variants from section 5, first with only the two shortest
windows, then with only the two longest. Which variant changes its rank?

In [ ]:
for wins in ((32, 64), (1024, 2048)):
    sub = [(w, b) for w, b in zip(MEL_WINDOWS, MEL_BANDS, strict=True) if w in wins]
    vals = {}
    for name, y in variants.items():
        vals[name] = sum(
            float((mel(y, w, b).clamp_min(1e-5).log10() - mel(x0, w, b).clamp_min(1e-5).log10()).abs().mean())
            for w, b in sub
        )
    print(f"windows {wins}: " + " < ".join(f"{n} {num(v, 2)}" for n, v in sorted(vals.items(), key=lambda kv: kv[1])))

## 11 · Limitations

- **Thousands of times smaller than the paper.** 1.2 million parameters instead of 1.6 billion; 21.9 minutes of
  English digits from 6 speakers instead of 3 million hours of speech, sound and music; 8 kHz instead of 24 kHz.
  Absolute quality can't be compared, and even at 32 layers the higher harmonics stay blurred.
- **No GAN, a single training stage.** The paper trains 520,000 steps without a GAN, then 500,000 steps with one (MPD
  and complex-spectrum discriminators). On our budget, 1,500 GAN steps did worse than the same compute spent on the
  mel loss, so we dropped the GAN. The mel loss doesn't look at phase, and it is the GAN part that teaches the decoder
  to build the right phases.
- **Not converged.** The validation mel distance is still falling at step 10,000. Every conclusion is about the same
  10,000-step budget, and η = 3 · 10⁻³ was chosen for the reference configuration and then shared by all of them.
- **Small judges.** Two CNNs trained on 2,397 recordings stand in for the paper's speech recognizer and its
  speaker-verification model. "The right voice" here means picking the right one of 6 known speakers, which is much
  easier than recognizing an arbitrary person.
- **The TTS is not zero-shot.** The paper copies the voices of people it has never heard. Here all 6 speakers are in
  the training set, and the text is a single digit. The voice results are therefore a generous upper bound, and they
  are still weak.
- **Two departures from the paper:** the spectral output (30 ms of delay; section 8 compares it with the paper's
  approach) and K̂ drawn from {1, …, 32} instead of {1, …, 31}.
- **3 seeds.** Differences whose seed bands overlap are reported as within the seed range.

## Summary

- **An audio tokenizer** has an encoder, a quantizer and a decoder. CAT cuts the waveform into 12.5 frames per
  second, with 32 tokens per frame and 10 bits per token: at most 4,000 bits/s, 96 times smaller than a 24 kHz wav
  file.
- **The measure has to behave like the ear.** Flipping the sign of the whole wave gives the largest MSE, yet the ear
  hears no difference; the multi-scale mel loss compares the magnitudes of the spectrum at several window lengths.
- **RVQ** quantizes the residual layer after layer, and the gradient passes through the argmax by straight-through.
  **Quantizer dropout** teaches the first layers to stand alone, so one tokenizer works at every bitrate.
- **The miniature CAT** (1,166,882 parameters): 1 layer (125 bits/s) is enough to hear the digit 81.9% of the time,
  the voice needs about 4 layers, and from 8 layers on it gets no better. The last layers use few codes: about
  3,300 bits/s is real information.
- **The claims:** A holds, with a trade-off: without dropout the tokenizer is the best at 32 layers and useless at 1.
  B holds: end-to-end training pulls well ahead of partial training. C holds: bigger models gain more at high
  bitrates, though the step from 64 to 96 is within the seed range. D holds: larger batches are better for the same
  number of steps. E depends: the semantic loss helps at 1 layer and costs quality at every bitrate. F holds for
  intelligibility: PSD keeps the TTS good when fewer layers are generated, but the voice is only partly copied at
  every p.

## Review questions

1. Flipping the sign of the whole wave gives an MSE / power of 4, yet a mel loss of 0. What does the mel loss ignore?
   Why is that both a strength and a gap?
2. The tokenizer trained without quantizer dropout, cut to 1 layer, gives a mel distance of 4.52, even though layer 1
   is trained on every example. Why?
3. A tokenizer runs at 12.5 frames per second with codebooks of 4,096 codes, and keeps 8 layers. How many bits per
   second is that? Flattened for a language model, how many tokens long is a 10-second utterance?
4. Why does CAT-TTS use a Temporal Transformer and a Depth Transformer instead of one Transformer reading all 32
   layers flattened?
5. The p = 0 TTS model generating at 1 layer fails to end 13.4% of its utterances on average. Why, when the
   "end of utterance" choice is right there in layer 1?
6. Experiment D fixes the number of steps and raises the batch size. Design an experiment that separates the effect
   of the batch size from the amount of data seen.

<details><summary>Hints and answers</summary>

1. The mel loss compares only the *magnitude* of the spectrum in each window, not the phase. A sign flip just adds π
   to the phase of every frequency, so every magnitude stays the same, and the ear can't hear it either. That is the
   strength: the model isn't punished for harmless differences. The gap: the mel loss doesn't teach the decoder to
   choose phases so that overlapping windows add up consistently. The paper adds GAN discriminators that look
   directly at the waveform and at the complex spectrum to take care of that.
2. When all 32 layers are always present, the decoder has only ever seen the sum of all 32, and nothing forces
   layer 1 to carry the most important part on its own, because the later layers are always there to correct it.
   Cutting down to 1 layer feeds the decoder a vector it has never seen.
3. log₂ 4,096 = 12 bits, so 12.5 · 8 · 12 = 1,200 bits/s. Flattened, that is 12.5 · 8 = 100 tokens per second, or
   1,000 tokens for 10 seconds.
4. Flattening 32 layers makes 10 seconds into 4,000 tokens, and the cost of attention grows with the square of the
   length. The Temporal Transformer sees only 125 positions for 10 seconds; the Depth Transformer is small, runs over
   32 positions inside one frame, and handles the relations between layers. The split also keeps PSD simple: drop the
   last positions of the Depth Transformer and some terms of the embedding sum.
5. The model predicts "end of utterance" from the Temporal Transformer's state, and that state is computed from the
   summed embeddings of the earlier frames. The p = 0 model only ever learned with the sum of all 32 layers; at 1
   layer that sum looks very different, so the cues for "done reading" are no longer where the model learned them.
   With PSD the model has seen exactly this kind of input during training.
6. For example: batch 8 for 40,000 steps against batch 32 for 10,000 steps, so both see the same 320,000 clips, with
   η chosen separately for each batch size on the validation set. Comparing at the same amount of data (and
   recording the wall-clock time as well) shows whether a large batch is really better per example, or only per step.
</details>

## Further reading

- Gong et al. (2026), *MOSS-Audio-Tokenizer: Scaling Audio Tokenizers for Future Audio Foundation Models*:
  [arXiv:2602.10934](https://arxiv.org/abs/2602.10934); code and weights:
  [OpenMOSS/MOSS-Audio-Tokenizer](https://github.com/OpenMOSS/MOSS-Audio-Tokenizer).
- van den Oord, Vinyals & Kavukcuoglu (2017), *Neural Discrete Representation Learning* (VQ-VAE):
  [arXiv:1711.00937](https://arxiv.org/abs/1711.00937).
- Zeghidour et al. (2021), *SoundStream: An End-to-End Neural Audio Codec*:
  [arXiv:2107.03312](https://arxiv.org/abs/2107.03312).
- Défossez et al. (2022), *High Fidelity Neural Audio Compression* (EnCodec):
  [arXiv:2210.13438](https://arxiv.org/abs/2210.13438).
- Kumar et al. (2023), *High-Fidelity Audio Compression with Improved RVQGAN* (DAC):
  [arXiv:2306.06546](https://arxiv.org/abs/2306.06546).
- Kong, Kim & Bae (2020), *HiFi-GAN*: [arXiv:2010.05646](https://arxiv.org/abs/2010.05646).
- Siuzdak (2023), *Vocos: Closing the gap between time-domain and Fourier-based neural vocoders*:
  [arXiv:2306.00814](https://arxiv.org/abs/2306.00814).
- Lee et al. (2022), *Autoregressive Image Generation using Residual Quantization* (RQ-Transformer):
  [arXiv:2203.01941](https://arxiv.org/abs/2203.01941).
- Défossez et al. (2024), *Moshi: a speech-text foundation model for real-time dialogue*:
  [arXiv:2410.00037](https://arxiv.org/abs/2410.00037).
- Kyutai (2025), *Neural audio codecs: how to get audio into LLMs*:
  [kyutai.org/codec-explainer](https://kyutai.org/codec-explainer/).
- Free Spoken Digit Dataset: [github.com/Jakobovski/free-spoken-digit-dataset](https://github.com/Jakobovski/free-spoken-digit-dataset).

**Back to** the course (in Vietnamese):
[Lesson 14 · Tokenization & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/) ·
[Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)